1. List all combos for each cage and do checks to make sure the combos work. Like each row, column, and 9x9 square should have the digits 1-9, and each cage should add up to the sum and ensure that there are no duplicates.
2. Fill any cage that have a single value possible
3. In case a cage has only one combo but unknown spots, cancel the numbers in that combo from the other cages if possible.
4. Do 45 rule, create new cages as necessary to ensure sum is 45 for all rows and columns, and 9x9 squares. Then revisit step 1.
5. Do multiples of 45 rule, going through each combo of sequential rows, columns, and then 9x9 squares and ensuring that they add up to multiples of 45 and creating new cages as needed. Then revisit step 1.
6. Do Innie and Outies rule, creating new cages on the outies from the innies made in rules #4 and #5. Then revisit step 1.


In [35]:
import pygame

class Cell:
    
    def __init__(self, row: int, col: int, box: int, cage: int, candidates: int, value: int|None = None):
        self.row = row
        self.col = col
        self.box = box
        self.cage = cage
        self.candidates = candidates
        self.value = value
        self.combos = []
        self.cages: list[Cage] = []
        
    def __repr__(self):
        return f"Cell(r={self.row},c={self.col},b={self.box},ca={self.cage},cand={bin(self.candidates)})"

class Cage:    
    def __init__(self, cage_sum: int, cells: list[Cell], perms: list[list[tuple[Cell, int]]], is_phantom_cage: bool):
        self.cage_sum = cage_sum
        self.cells = cells
        self.perms = perms
        self.is_phantom_cage = is_phantom_cage
        self.is_filled = False
        
    def __repr__(self):
        return f"Cage(sum={self.cage_sum},cells={[str(c) for c in self.cells]},perms={self.perms},phantom={self.is_phantom_cage},filled={self.is_filled})"
    
class KillerSudokuBoard:    
    def __init__(self, cage_data: list[list[int|list[tuple[int, int]]]], reverse_coords: bool = False):
        self.cells: list[Cell] = []
        self.cages: list[Cage] = []
        self.grid: dict[tuple[int, int], Cell] = {}
        self.create_cages_and_cells_from_cage_data(cage_data, reverse_coords)
        
    def create_cages_and_cells_from_cage_data(self, cage_data: list[list[int|list[tuple[int, int]]]], reverse_coords: bool):
        for cage_sum, cell_coords in cage_data:
            cage = Cage(cage_sum=cage_sum, cells=[], perms=[], is_phantom_cage=False)
            for row, col in cell_coords:
                if reverse_coords:
                    row, col = col, row
                box = (row // 3) * 3 + (col // 3)
                cell = Cell(row=row, col=col, box=box, cage=len(self.cages), candidates=0b0)
                cell.cages.append(cage)
                cage.cells.append(cell)
                self.cells.append(cell)
                self.grid[(row, col)] = cell
            self.cages.append(cage)
            
    def verify_cells(self) -> dict[str, list[tuple[int,int]]]:

        expected = {(r,c) for r in range(9) for c in range(9)}

        found: set[tuple[int, int]] = set()
        duplicates: set[tuple[int, int]] = set()

        for cell in self.cells:

            coord = (cell.row, cell.col)

            if coord in found:
                duplicates.add(coord)

            found.add(coord)

        missing = expected - found

        return {
            "missing": sorted(missing),
            "duplicates": sorted(duplicates)
        }
            
    def copy(self):

        new_board = KillerSudokuBoard.__new__(KillerSudokuBoard)

        new_board.cells = []
        new_board.cages = []

        cell_map = {}

        # Copy cells first
        for cell in self.cells:

            new_cell = Cell(
                row=cell.row,
                col=cell.col,
                box=cell.box,
                cage=cell.cage,
                candidates=cell.candidates
            )

            new_board.cells.append(new_cell)

            cell_map[(cell.row,cell.col)] = new_cell

        # Copy cages
        for cage in self.cages:

            new_cells = []

            for cell in cage.cells:

                new_cells.append(
                    cell_map[(cell.row,cell.col)]
                )

            new_perms = [
                perm.copy()
                for perm in cage.perms
            ]

            new_cage = Cage(
                cage_sum=cage.cage_sum,
                cells=new_cells,
                perms=new_perms,
                is_phantom_cage=cage.is_phantom_cage
            )

            new_board.cages.append(new_cage)

        return new_board
        

In [90]:
CELL_SIZE = 90
GAP = 5
STEP = CELL_SIZE + GAP
BOARD_SIZE = STEP * 9 + GAP
CAND_SIZE_X = CELL_SIZE // 3
CAND_SIZE_Y = CELL_SIZE // 6
X_OFFSET = 10
Y_OFFSET = 10

BLUE = (0,0,255)

class PyGameKillerSudokuBoard(KillerSudokuBoard):
    def __init__(self, cage_data: list[list[int|list[tuple[int, int]]]], reverse_coords: bool = False):
        super().__init__(cage_data, reverse_coords)
    
    def draw(self, screen: pygame.Surface):

        font_sum = pygame.font.SysFont("arial", 18)
        font_cand = pygame.font.SysFont("arial", 14)
        font_val = pygame.font.SysFont("arial", 36)

        # Build lookup grid
        grid = [[None for _ in range(9)] for _ in range(9)]

        for cell in self.cells:
            grid[cell.row][cell.col] = cell
        # print(grid)
        # Draw background
        screen.fill((255,255,255))

        # Draw cells + candidates
        for row in range(9):
            for col in range(9):

                cell = grid[row][col]

                x = col * CELL_SIZE + GAP
                y = row * CELL_SIZE + GAP 

                pygame.draw.rect(
                    screen,
                    (240,240,240),
                    (x + X_OFFSET,y+Y_OFFSET,CELL_SIZE,CELL_SIZE)
                )

                # Draw value
                if cell.value is not None:
                    text = font_val.render(str(cell.value),True,BLUE)
                    text_rect = text.get_rect(center=(x + CELL_SIZE//2 + X_OFFSET, y + CELL_SIZE//2 + Y_OFFSET))
                    screen.blit(text, text_rect)
                else:
                    # Draw candidates
                    for n in range(1,10):
                                                
                        if cell.candidates & (1<<(n-1)):

                            cx = (n-1) % 3
                            cy = (n-1) // 3
                            
                            tx = x + cx*CAND_SIZE_X + 8 + X_OFFSET
                            ty = y + (cy+2)*CAND_SIZE_Y + 5 + Y_OFFSET

                            text = font_cand.render(str(n),True,(0,0,0))
                            screen.blit(text,(tx,ty))

        # Draw grid lines
        for i in range(10):

            thickness = 3 if i%3==0 else 2

    
            gap_offset = 4*GAP//2

            pygame.draw.line(
                screen,
                (0,0,0),
                (0,i*(CELL_SIZE)+gap_offset),
                (BOARD_SIZE,i*(CELL_SIZE)+gap_offset),
                thickness
            )

            pygame.draw.line(
                screen,
                (0,0,0),
                (i*(CELL_SIZE)+gap_offset,0),
                (i*(CELL_SIZE)+gap_offset,BOARD_SIZE),
                thickness
            )

        # Draw cage sums
        for cage in self.cages:

            top = min(cage.cells,key=lambda c:(c.row,c.col))

            x = top.col * CELL_SIZE + 3 + X_OFFSET + GAP
            y = top.row * CELL_SIZE + 1 + Y_OFFSET + GAP

            text = font_sum.render(str(cage.cage_sum),True,(0,0,0))

            screen.blit(text,(x,y))

        # Draw cage borders
        # Need to draw 2 lines per border - one for each cell that shares the border - to get the dashed effect
        for cage in self.cages:
            if not cage.is_phantom_cage:
                for cell in cage.cells:

                    x = cell.col * CELL_SIZE + X_OFFSET
                    y = cell.row * CELL_SIZE + Y_OFFSET

                    neighbors = {
                        (cell.row-1,cell.col),
                        (cell.row+1,cell.col),
                        (cell.row,cell.col-1),
                        (cell.row,cell.col+1)
                    }

                    cage_cells = {(c.row,c.col) for c in cage.cells}

                    dash = 4

                    # Top
                    if (cell.row-1,cell.col) not in cage_cells:
                        for d in range(0,CELL_SIZE,8):
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+d,y+GAP),
                                (x+d+dash,y+GAP),
                                2
                            )
                            
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+d,y-GAP),
                                (x+d+dash,y-GAP),
                                2
                            )

                    # Bottom
                    if (cell.row+1,cell.col) not in cage_cells:
                        for d in range(0,CELL_SIZE,8):
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+d,y+CELL_SIZE+GAP),
                                (x+d+dash,y+CELL_SIZE+GAP),
                                2
                            )
                            
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+d,y+CELL_SIZE-GAP),
                                (x+d+dash,y+CELL_SIZE-GAP),
                                2
                            )


                    # Left
                    if (cell.row,cell.col-1) not in cage_cells:
                        for d in range(0,CELL_SIZE,8):
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+GAP,y+d),
                                (x+GAP,y+d+dash),
                                2
                            )
                            
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x-GAP,y+d),
                                (x-GAP,y+d+dash),
                                2
                            )

                    # Right
                    if (cell.row,cell.col+1) not in cage_cells:
                        for d in range(0,CELL_SIZE,8):
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+CELL_SIZE+GAP,y+d),
                                (x+CELL_SIZE+GAP,y+d+dash),
                                2
                            )
                            
                            pygame.draw.line(
                                screen,(0,0,0),
                                (x+CELL_SIZE-GAP,y+d),
                                (x+CELL_SIZE-GAP,y+d+dash),
                                2
                            )

In [4]:
cage_data: list[list[int|list[tuple[int, int]]]] = [
[13, [(0,0),(0,1)]],

[10, [(1,0),(1,1)]],

[14, [(2,0),(3,0), (2,1)]],

[23, [(4,0),(5,0), (6,0),(5,1)]],

[28, [(7,0),(8,0),(6,1),(7,1), (8,1), (7, 2)]],

[3, [(3,1),(4,1)]],

[7, [(0,2),(0,3)]],

[18, [(1,2),(1,3),(1,4)]],

[5, [(2,2),(3,2)]],

[16, [(4,2),(4,3)]],

[25, [(5,2),(6,2),(5,3),(6,3)]],

[23, [(8,2),(7,3),(8,3),(8,4)]],

[17, [(2,3),(3,3),(2,4)]],

[9, [(0,4),(0,5),(1,5)]],

[20, [(3,4),(4,4),(3,5),(4,5)]],

[5, [(5,4),(6,4)]],

[12, [(7,4),(7,5)]],

[16, [(2,5),(2,6),(3,6)]],

[15, [(5,5),(4,6),(5,6)]],

[12, [(6,5),(6,6)]],

[14, [(8,5),(7,6),(8,6)]],

[44, [(0,6),(1,6),(0,7),(1,7),(2,7),(0,8),(1,8),(2,8)]],

[19, [(3,7),(3,8),(4, 8)]],

[9, [(4,7),(5,7), (6,7)]],

[11, [(7,7),(8,7)]],

[8, [(5,8),(6,8)]],

[9, [(7,8),(8,8)]]
]

In [5]:
board = PyGameKillerSudokuBoard(cage_data, reverse_coords=True)


In [6]:
board.verify_cells()




{'missing': [], 'duplicates': []}

In [37]:
from itertools import combinations, permutations

ALL_COMBOS: dict[tuple[int,int], list[list[int]]] = {}
for size in range(1,10):
    for combo_sum in range(1,46):
        ALL_COMBOS[(size,combo_sum)] = []

        digits = range(1,10)

        for combo in combinations(digits, size):
            if sum(combo) == combo_sum:
                ALL_COMBOS[(size,combo_sum)].append(list(combo))

In [8]:
ALL_COMBOS

{(1, 1): [[1]],
 (1, 2): [[2]],
 (1, 3): [[3]],
 (1, 4): [[4]],
 (1, 5): [[5]],
 (1, 6): [[6]],
 (1, 7): [[7]],
 (1, 8): [[8]],
 (1, 9): [[9]],
 (1, 10): [],
 (1, 11): [],
 (1, 12): [],
 (1, 13): [],
 (1, 14): [],
 (1, 15): [],
 (1, 16): [],
 (1, 17): [],
 (1, 18): [],
 (1, 19): [],
 (1, 20): [],
 (1, 21): [],
 (1, 22): [],
 (1, 23): [],
 (1, 24): [],
 (1, 25): [],
 (1, 26): [],
 (1, 27): [],
 (1, 28): [],
 (1, 29): [],
 (1, 30): [],
 (1, 31): [],
 (1, 32): [],
 (1, 33): [],
 (1, 34): [],
 (1, 35): [],
 (1, 36): [],
 (1, 37): [],
 (1, 38): [],
 (1, 39): [],
 (1, 40): [],
 (1, 41): [],
 (1, 42): [],
 (1, 43): [],
 (1, 44): [],
 (1, 45): [],
 (2, 1): [],
 (2, 2): [],
 (2, 3): [[1, 2]],
 (2, 4): [[1, 3]],
 (2, 5): [[1, 4], [2, 3]],
 (2, 6): [[1, 5], [2, 4]],
 (2, 7): [[1, 6], [2, 5], [3, 4]],
 (2, 8): [[1, 7], [2, 6], [3, 5]],
 (2, 9): [[1, 8], [2, 7], [3, 6], [4, 5]],
 (2, 10): [[1, 9], [2, 8], [3, 7], [4, 6]],
 (2, 11): [[2, 9], [3, 8], [4, 7], [5, 6]],
 (2, 12): [[3, 9], [4, 8], [5, 7]

In [ ]:
class KillerSudokuSolver:
    def __init__(self, board: KillerSudokuBoard):
        self.board = board
        self.new_cages: list[Cage] = self.board.cages
        
    def set_valid_candidates(self, cell: Cell):
        # This function will set the valid candidates for a cell based on the current combos for the cages that this cell is in
        valid_candidates = 0b0
        for cage in cell.cages:
            for combo in cage.perms:
                for c, value in combo:
                    if c == cell:
                        if cell.value is None:
                            valid_candidates |= (1 << (value - 1))
        cell.candidates = valid_candidates
        
    def revalidate_cage_permutations(self, current_cage: Cage, updated_cage: Cage) -> list[Cage]:
        # This function will revalidate the permutations for the cage that share cells with the updated cage
        if current_cage == updated_cage:
            return []
        print(f"Revalidating cage with sum {current_cage.cage_sum} based on updated cage with sum {updated_cage.cage_sum}...")
        affected_cages: list[Cage] = []
        for perm in current_cage.perms:
            valid_perm = True
            for cell, value in perm:
                if cell in updated_cage.cells:
                    # Check if the value for this cell in the combo is consistent with any combo in the updated cage
                    if not any(
                        value == v and c == cell
                        for updated_perm in updated_cage.perms
                        for c, v in updated_perm
                    ):
                        valid_perm = False
                        break
            if not valid_perm:
                current_cage.perms.remove(perm)
                affected_cages.append(current_cage)
                # We just updated the combos for the current cage, so we need to revalidate any cages that share cells with this cage as well
                for cell in current_cage.cells:
                    for other_cage in cell.cages:
                        if other_cage != current_cage:
                            other_affected_cages = self.revalidate_cage_permutations(other_cage, current_cage)
                            affected_cages.extend(other_affected_cages)
        return affected_cages
        
          
    
    def generate_perms_for_cage(self, cage: Cage):
        # This is a helper function that will generate permutations for a single cage
        # It will consider the candidates for each cell and the sum constraint
        size = len(cage.cells)
        target = cage.cage_sum
        
        combos = ALL_COMBOS.get((size, target), [])
        valid_combos: list[list[tuple[Cell, int]]] = []
        
        for combo in combos:
            # Need to check each row, column, and box for candidate conflicts
            # Go through all combos of assigning these numbers to the cells
            for perm in permutations(combo):
                valid_perm = True
                perm_in_combo_form = list(zip(cage.cells, perm))
                # First, check all combos created from other cages that share this cell - if the value in the cell is not in any of the combos for the other cage, then this combo is not valid
                for cell, value in perm_in_combo_form:
                    # First get all cages that share this cell
                    for other_cage in cell.cages:
                        if other_cage == cage:
                            continue
                        # Check if any combo in the other cage has this value for this cell
                        if not any(
                            value == v and c == cell
                            for combo in other_cage.perms
                            for c, v in combo
                        ):
                            valid_perm = False
                            break
                    if not valid_perm:
                        break
                    
                    # Next, check if any other cell in the same row, column, or box has this value - if not, then this combo is not valid
                    for other_cell in self.board.cells:
                        if other_cell == cell:
                            continue
                        if other_cell.row == cell.row or other_cell.col == cell.col or other_cell.box == cell.box:
                            if value == other_cell.value:
                                valid_perm = False
                                break
                    if not valid_perm:
                        break
                if valid_perm:
                    valid_combos.append(perm_in_combo_form)
        cage.perms = valid_combos
        # Need to update the combos for any cages that share cells with this cage, since they may now have new constraints based on the valid combos for this cage
        affected_cages = [cage]
        for cell in cage.cells:
            for other_cage in cell.cages:
                if other_cage != cage:
                    other_affected_cages = self.revalidate_cage_permutations(other_cage, cage)
                    affected_cages.extend(other_affected_cages)

        # After revalidating the combos for the cages that share cells with this cage, we need to update the valid candidates for the cells in this cage as well
        for affected_cage in affected_cages:
            for cell in affected_cage.cells:
                self.set_valid_candidates(cell)                    
        
        
    def generate_perms_for_new_cages(self):
        # This function will generate all valid combinations of numbers for a new cage
        # It should consider the sum, the number of cells, and the candidates for each cell
        for cage in self.new_cages:
            self.generate_perms_for_cage(cage)
        
        updated_cages = self.new_cages
        self.new_cages = []
        return updated_cages
    
    def apply_single_values(self):
        """This function will look for any cells that have only a single valid candidate - if it finds any, it will set the value for that cell and return the updated cages"""
        updated_cages: list[Cage] = []
        affected_cages: list[Cage] = []
        for cell in self.board.cells:
            if cell.value is not None:
                continue
            if bin(cell.candidates).count("1") == 1:
                value = (cell.candidates & -cell.candidates).bit_length()
                cell.value = value
                print(f"Applying single value for cell ({cell.row}, {cell.col}): {value}")
                for cage in cell.cages:
                    updated_cage = False
                    for perm in cage.perms:
                        if any(c == cell and v == value for c, v in perm):
                            continue
                        else:
                            cage.perms.remove(perm)
                            updated_cage = True
                    if updated_cage:
                        updated_cages.append(cage)
                        other_affected_cages = self.revalidate_cage_permutations(cage, cage)
                        affected_cages.extend(other_affected_cages)
                        
            # We need to go through and check if any other cell in the same row, column, or box has this value in a potential permutation, and if so, we need to remove that permutation as well
            row = cell.row
            col = cell.col
            box = cell.box
            value = cell.value
            # Go through rows
            for current_col in range(9):
                if current_col != col:
                    other_cell = self.board.grid[(row, current_col)]
                    for other_cage in other_cell.cages:
                        updated_cage = False
                        new_perms: list[list[tuple[Cell, int]]] = []
                        for perm in other_cage.perms:
                            if any(c == other_cell and v == value for c, v in perm):
                                updated_cage = True
                            else:
                                new_perms.append(perm)
                        other_cage.perms = new_perms
                        if updated_cage:
                            affected_cages.append(other_cage)
                            other_affected_cages = self.revalidate_cage_permutations(other_cage, other_cage)
                            affected_cages.extend(other_affected_cages)
            # Go through columns
            for current_row in range(9):
                if current_row != row:
                    other_cell = self.board.grid[(current_row, col)]
                    for other_cage in other_cell.cages:
                        updated_cage = False
                        new_perms: list[list[tuple[Cell, int]]] = []
                        for perm in other_cage.perms:
                            if any(c == other_cell and v == value for c, v in perm):
                                updated_cage = True
                            else:
                                new_perms.append(perm)
                        other_cage.perms = new_perms
                        if updated_cage:
                            affected_cages.append(other_cage)
                            other_affected_cages = self.revalidate_cage_permutations(other_cage, other_cage)
                            affected_cages.extend(other_affected_cages)
            # Go through boxes
            for current_row in range((box // 3) * 3, (box // 3) * 3 + 3):
                for current_col in range((box % 3) * 3, (box % 3) * 3 + 3):
                    if current_row != row and current_col != col:
                        other_cell = self.board.grid[(current_row, current_col)]
                        for other_cage in other_cell.cages:
                            updated_cage = False
                            new_perms: list[list[tuple[Cell, int]]] = []
                            for perm in other_cage.perms:
                                if any(c == other_cell and v == value for c, v in perm):
                                    updated_cage = True
                                else:
                                    new_perms.append(perm)
                            other_cage.perms = new_perms
                            if updated_cage:
                                affected_cages.append(other_cage)
                                other_affected_cages = self.revalidate_cage_permutations(other_cage, other_cage)
                                affected_cages.extend(other_affected_cages)
                                
        # After revalidating the combos for the cages that share cells with this cage, we need to update the valid candidates for the cells in this cage as well
        for affected_cage in updated_cages+affected_cages:
            for cell in affected_cage.cells:
                self.set_valid_candidates(cell) 
        return updated_cages, affected_cages
    
    def apply_single_permutations(self):
        # This function will look for any cages that have only a single valid permutation - if it finds any, it will set the values for the cells in that permutation and return the updated cages
        updated_cages: list[Cage] = []
        affected_cages: list[Cage] = []
        print("Called apply_single_permutations")
        for cage in self.board.cages:
            if cage.is_filled:
                continue
            if len(cage.perms) == 1:
                print(f"Applying single permutation for cage with sum {cage.cage_sum}")
                perm = cage.perms[0]
                for cell, value in perm:
                    cell.value = value
                updated_cages.append(cage)
                for other_cage in cell.cages:
                    other_affected_cages: list[Cage] = self.revalidate_cage_permutations(other_cage, cage)
                    affected_cages.extend(other_affected_cages)
                cage.is_filled = True
                
        # After revalidating the combos for the cages that share cells with this cage, we need to update the valid candidates for the cells in this cage as well
        for affected_cage in updated_cages+affected_cages:
            for cell in affected_cage.cells:
                self.set_valid_candidates(cell) 
        return updated_cages, affected_cages
    
    def handle_single_combos_in_region(self):
        # This function will look for singular combos contained in a row, column, or box - if it finds any, it will remove any perms that have overlapping values for the cell in that combo and return the updated cages
        affected_cages: list[Cage] = []
        for cage in self.board.cages:
            # print(f"Checking cage with sum {cage.cage_sum} for single combos in region...")
            # check if this cage has any combos that are the only combo in their row, column, or box
            is_single_combo_in_row = False
            is_single_combo_in_col = False
            is_single_combo_in_box = False
            
            # Check if cage has only one value
            if len(cage.cells) == 1:
                is_single_combo_in_row = True
                is_single_combo_in_col = True
                is_single_combo_in_box = True
            
            # Otherwise, need to check if the entire cage is part of a row, column, or box
            else:
                rows: set[int] = set()
                cols: set[int]  = set()
                boxes: set[int]  = set()
                for cell in cage.cells:
                    rows.add(cell.row)
                    cols.add(cell.col)
                    boxes.add(cell.box)
                if len(rows) == 1:
                    is_single_combo_in_row = True
                if len(cols) == 1:
                    is_single_combo_in_col = True
                if len(boxes) == 1:
                    is_single_combo_in_box = True
            
            # Now check if we have a singular combo (there can still be many perms, they just must all have the same value)
            unique_values_in_perms: set[int] = set()
            is_singular_combo = False
            for perm in cage.perms:
                for _, value in perm:
                    unique_values_in_perms.add(value)
            if len(unique_values_in_perms) == len(cage.cells):
                is_singular_combo = True
            
            if is_singular_combo and (is_single_combo_in_row or is_single_combo_in_col or is_single_combo_in_box):
                # We have a singular combo that is the only combo in its row, column, or box - we can remove any perms from other cages that share cells in this region that have overlapping values for the cell in this combo
                perm = cage.perms[0]
                values_to_remove_form_other_cells: set[int] = set()
                first_set = False
                row, col, box = None, None, None
                for cell, value in perm:
                    values_to_remove_form_other_cells.add(value)
                    if not first_set:
                        row = cell.row
                        col = cell.col
                        box = cell.box
                        first_set = True
                    
                # Now go through and remove any perms from other cages that share cells in this region that have overlapping values for the cell in this combo
                if is_single_combo_in_row:
                    for current_col in range(9):
                        cell = self.board.grid[(row, current_col)]
                        for other_cage in cell.cages:
                            if other_cage != cage:
                                # We first need to check if this cage overlaps with this cage, in which case we should not remove any perms
                                cage_cells = set(cage.cells)
                                other_cage_cells = set(other_cage.cells)
                                if cage_cells.isdisjoint(other_cage_cells):
                                    # This cage does not overlap with the current cage, so we can remove any perms that have overlapping values for the cell in this combo
                                    updated_cage = False
                                    # print(f"Checking cage with sum {other_cage.cage_sum} for overlapping values in row {row}")
                                    if any(value in values_to_remove_form_other_cells for other_perm in other_cage.perms for cell, value in other_perm if cell.row == row):
                                        # print(f"Found overlapping value in cage with sum {other_cage.cage_sum} for row {row}, removing perms...")
                                        updated_cage = True
                                        affected_cages.append(other_cage)
                                        other_cage.perms = [perm for perm in other_cage.perms if not any(value in values_to_remove_form_other_cells for cell, value in perm if cell.row == row)]
                                    if updated_cage:
                                        other_affected_cages = self.revalidate_cage_permutations(other_cage, cage)
                                        affected_cages.extend(other_affected_cages)
                
                if is_single_combo_in_col:
                    for current_row in range(9):
                        cell = self.board.grid[(current_row, col)]
                        for other_cage in cell.cages:
                            if other_cage != cage:
                                # We first need to check if this cage overlaps with this cage, in which case we should not remove any perms
                                cage_cells = set(cage.cells)
                                other_cage_cells = set(other_cage.cells)
                                if cage_cells.isdisjoint(other_cage_cells):
                                    # This cage does not overlap with the current cage, so we can remove any perms that have overlapping values for the cell in this combo
                                    updated_cage = False
                                    # print(f"Checking cage with sum {other_cage.cage_sum} for overlapping values in col {col}")
                                    if any(value in values_to_remove_form_other_cells for other_perm in other_cage.perms for cell, value in other_perm if cell.col == col):
                                        # print(f"Found overlapping value in cage with sum {other_cage.cage_sum} for col {col}, removing perms...")
                                        updated_cage = True
                                        affected_cages.append(other_cage)
                                        other_cage.perms = [perm for perm in other_cage.perms if not any(value in values_to_remove_form_other_cells for cell, value in perm if cell.col == col)]
                                    if updated_cage:
                                        other_affected_cages = self.revalidate_cage_permutations(other_cage, cage)
                                        affected_cages.extend(other_affected_cages)
                                        
                if is_single_combo_in_box:
                    for current_row in range((box//3)*3, (box//3)*3 + 3):
                        for current_col in range((box%3)*3, (box%3)*3 + 3):
                            cell = self.board.grid[(current_row, current_col)]
                            for other_cage in cell.cages:
                                if other_cage != cage:
                                    # We first need to check if this cage overlaps with this cage, in which case we should not remove any perms
                                    cage_cells = set(cage.cells)
                                    other_cage_cells = set(other_cage.cells)
                                    if cage_cells.isdisjoint(other_cage_cells):
                                        # This cage does not overlap with the current cage, so we can remove any perms that have overlapping values for the cell in this combo
                                        updated_cage = False
                                        # print(f"Checking cage with sum {other_cage.cage_sum} for overlapping values in box {box}")
                                        if any(value in values_to_remove_form_other_cells for other_perm in other_cage.perms for cell, value in other_perm if cell.box == box):
                                            # print(f"Found overlapping value in cage with sum {other_cage.cage_sum} for box {box}, removing perms...")
                                            updated_cage = True
                                            affected_cages.append(other_cage)
                                            other_cage.perms = [perm for perm in other_cage.perms if not any(value in values_to_remove_form_other_cells for cell, value in perm if cell.box == box)]
                                        if updated_cage:
                                            other_affected_cages = self.revalidate_cage_permutations(other_cage, cage)
                                            affected_cages.extend(other_affected_cages)
        
        # After revalidating the combos for the cages that share cells with this cage, we need to update the valid candidates for the cells in this cage as well
        for affected_cage in affected_cages:
            for cell in affected_cage.cells:
                self.set_valid_candidates(cell) 
                                
                
        return affected_cages
            
    def generate_innies_and_outies(self):
        """This function will generate cages based off of each sequential set of rows, columns, and boxes and assign them to the new_cages attribute - these cages will then be processed in the next step"""
        for window_size in range(1, 10):
            for start_row in range(0, 10 - window_size):
                cells_in_window = [self.board.grid[(row, col)] for row in range(start_row, start_row + window_size) for col in range(9)]
                # Now that we have the cells in the window, we need to find the cages that these cells are a part of (only the non-phantom cages)
                overlapping_cages: set[Cage] = set()
                for cell in cells_in_window:
                    for cage in cell.cages:
                        if not cage.is_phantom_cage:
                            overlapping_cages.add(cage)
                # For cages that are partially in the window, we need to create a new cage for the cells in the window and a new cage for the cells outside the window
                innies: list[Cell] = []
                outies: list[Cell] = []
                for cage in overlapping_cages:
                    cells_in_cage = set(cage.cells)
                    cells_in_window_set = set(cells_in_window)
                    if not cells_in_cage.issubset(cells_in_window_set):
                        for cell in cage.cells:
                            if cell in cells_in_window_set:
                                innies.append(cell)
                            else:
                                outies.append(cell)
                if innies and outies:
                    print("We found innies and outies for rows {}-{}: {} innies and {} outies".format(start_row, start_row + window_size - 1, len(innies), len(outies)))
                    fully_contained_cage_sum = sum(cell.value for cell in cells_in_window if cell not in innies)
                    expected_sum = 45 * window_size
                    innies_cage_sum = expected_sum - fully_contained_cage_sum
                    innie_cage = Cage(innies_cage_sum, innies, perms=[], is_phantom_cage=True)
    
    def solve_next_step(self):
        # First, we need to generate combos for any cages that don't have them yet
        print("Solving next step...")
        if self.new_cages:
            print(f"Generating combos for {len(self.new_cages)} new cages...")
            return self.generate_perms_for_new_cages()
        
        print("Applying single values...")
        updated_cages, affected_cages = self.apply_single_values()
        if updated_cages or affected_cages:
            print(f"Applied single values for {len(updated_cages)} cages...")
            return updated_cages, affected_cages
        
        print("Applying single permutations...")
        updated_cages, affected_cages = self.apply_single_permutations()
        if updated_cages or affected_cages:
            print(f"Applied single permutations for {len(updated_cages)} cages...")
            return updated_cages, affected_cages
        
        print("Handling single combos in region...")
        updated_cages = self.handle_single_combos_in_region()
        if updated_cages:
            print(f"Handled single combos in region for {len(updated_cages)} cages...")
            return updated_cages
        
        print("Generating innies and outies...")
        updated_cages = self.generate_innies_and_outies()
        if updated_cages:
            print(f"Generated innies and outies for {len(updated_cages)} cages...")
            return updated_cages
        
        return None

In [59]:
killer_sudoku_solver = KillerSudokuSolver(board)

In [ ]:

board = PyGameKillerSudokuBoard(cage_data, reverse_coords=True)
killer_sudoku_solver = KillerSudokuSolver(board)

pygame.init()

LOG_AREA_WIDTH = 400
LOG_AREA_HEIGHT = 100
LOG_AREA_X_OFFSET = 20
LOG_AREA_Y_OFFSET = 20

screen = pygame.display.set_mode((BOARD_SIZE+ LOG_AREA_WIDTH + LOG_AREA_X_OFFSET, BOARD_SIZE + LOG_AREA_HEIGHT + LOG_AREA_Y_OFFSET))

running = True
start_time = pygame.time.get_ticks()
SOLVE_INTERVAL = 1000  # Solve every 1000 milliseconds (1 second)

# Colors
WHITE = (255, 255, 255)
RED = (255, 0, 0)
BLACK = (0, 0, 0)


font = pygame.font.SysFont("arial", 18)

# Clock for controlling FPS
clock = pygame.time.Clock()
first_draw = True
while running:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if first_draw:
        board.draw(screen)
        pygame.display.flip()
        first_draw = False
    
    current_time = pygame.time.get_ticks()
    elapsed_time = current_time - start_time
    if elapsed_time >= SOLVE_INTERVAL:
        pygame.draw.rect(
                    screen,
                    (255,0,0),
                    (LOG_AREA_X_OFFSET, BOARD_SIZE + LOG_AREA_Y_OFFSET,LOG_AREA_WIDTH,LOG_AREA_HEIGHT)
                )
        step_results = killer_sudoku_solver.solve_next_step()
        

        log_str = f"Step results: {step_results}"
        # print(log_str)
        # text_surface = font.render(log_str, True, (0, 0, 0))
        # Draw text on screen
        # screen.blit(text_surface, (LOG_AREA_X_OFFSET, BOARD_SIZE + LOG_AREA_Y_OFFSET))
        start_time = current_time  # Reset the timer after solving a step
    
    board.draw(screen)
    
    pygame.display.flip()

pygame.quit()

Solving next step...
Generating combos for 27 new cages...
Solving next step...
Applying single values...
Applying single permutations...
Called apply_single_permutations
Handling single combos in region...
Revalidating cage with sum 10 based on updated cage with sum 3...
Revalidating cage with sum 14 based on updated cage with sum 3...
Revalidating cage with sum 23 based on updated cage with sum 3...
Revalidating cage with sum 28 based on updated cage with sum 3...
Revalidating cage with sum 14 based on updated cage with sum 3...
Revalidating cage with sum 23 based on updated cage with sum 3...
Revalidating cage with sum 5 based on updated cage with sum 3...
Revalidating cage with sum 25 based on updated cage with sum 3...
Revalidating cage with sum 23 based on updated cage with sum 16...
Revalidating cage with sum 20 based on updated cage with sum 16...
Revalidating cage with sum 15 based on updated cage with sum 16...
Revalidating cage with sum 19 based on updated cage with sum 16..

KeyboardInterrupt: 